# CourtVision -- entrainement YOLO11s (balle de tennis) sur le dataset RacketVision

**Avant de lancer :** menu *Execution* > *Modifier le type d'execution* > **GPU T4** > Enregistrer.
Ensuite : *Execution* > **Tout executer**. Accepte l'acces a Google Drive quand la fenetre s'ouvre.

- Le dataset est reconstruit ici depuis Hugging Face (~30-40 min), rien a envoyer depuis ton PC.
- Entrainement plafonne a **10 h** ; les poids sont enregistres a chaque epoque dans
  `Mon Drive/CourtVision/runs/ball_rv/weights/` (`best.pt` = le meilleur).
- **Si Colab se deconnecte** : reviens sur le notebook, reverifie le GPU, puis *Tout executer* :
  l'entrainement reprend a la derniere epoque enregistree.
- Laisse l'onglet ouvert (Colab gratuit coupe une session inactive).

In [ ]:
# 1. Verifier qu'on a bien un GPU (sinon : Execution > Modifier le type d'execution > GPU T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive (pour garder les poids si la session coupe) + Ultralytics
#    Dans la fenetre Google : choisis ton compte puis clique "Tout selectionner" avant "Continuer".
import os
from google.colab import drive
try:
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/CourtVision'
except Exception as e:
    # Plan B : sans Drive, tout reste dans la session Colab (pas de reprise si elle coupe).
    print('!! Drive non monte (' + str(e)[:80] + ') -> sauvegarde locale dans /content/CourtVision')
    print('!! A la fin, telecharge best.pt via le panneau Fichiers (icone dossier a gauche).')
    DRIVE = '/content/CourtVision'
os.makedirs(DRIVE, exist_ok=True)
!pip -q install -U ultralytics huggingface_hub

In [ ]:
# 3. Dataset. Si tu as mis toi-meme yolo_rv.zip dans Mon Drive/CourtVision, il est utilise ;
#    sinon on telecharge les videos tennis RacketVision (1,8 Go) et on extrait les frames annotees.
import os, time, zipfile
DATA = '/content/yolo_rv'
ZIP = f'{DRIVE}/yolo_rv.zip'
if os.path.exists(f'{DATA}/data.yaml'):
    print('Dataset deja pret')
elif os.path.exists(ZIP):
    zipfile.ZipFile(ZIP).extractall('/content'); print('Dataset copie depuis Drive')
else:
    from huggingface_hub import snapshot_download
    for i in range(10):   # Hugging Face limite parfois le debit : on reessaie
        try:
            snapshot_download('linfeng302/RacketVision', repo_type='dataset', local_dir='/content/rv',
                              allow_patterns=['tennis/info/*', 'tennis/videos/*', 'tennis/all/*/csv/*'],
                              max_workers=4)
            break
        except Exception as e:
            print('nouvel essai dans 60 s :', str(e)[:120]); time.sleep(60)
    print('Telechargement termine')

In [ ]:
%%writefile /content/prepare.py
# Meme conversion que scripts/prepare_racketvision_yolo.py sur le PC.
import json, os, sys
from multiprocessing import Pool
import cv2, pandas as pd

SRC, OUT, BOX = '/content/rv/tennis', '/content/yolo_rv', 16   # boite de 16 px sur 1920x1080
OUT_W, OUT_H = 1280, 720

def convert_clip(job):
    split, match, rally = job
    df = pd.read_csv(f'{SRC}/all/{match}/csv/{rally}_ball.csv').fillna(0)
    labels = {int(r.Frame): r for r in df.itertuples()}
    cap = cv2.VideoCapture(f'{SRC}/videos/{match}_{rally}.mp4')
    vw, vh = int(cap.get(3)), int(cap.get(4))
    last, fid, n = max(labels), 0, 0
    while fid <= last:
        ok, frame = cap.read()
        if not ok:
            break
        if fid in labels:
            r, name = labels[fid], f'{match}_{rally}_{fid:04d}'
            cv2.imwrite(f'{OUT}/images/{split}/{name}.jpg',
                        cv2.resize(frame, (OUT_W, OUT_H), interpolation=cv2.INTER_AREA),
                        [cv2.IMWRITE_JPEG_QUALITY, 92])
            line = ''
            if r.Visibility == 1 and (r.X > 0 or r.Y > 0):
                line = f'0 {r.X / vw:.6f} {r.Y / vh:.6f} {BOX / vw:.6f} {BOX / vh:.6f}\n'
            open(f'{OUT}/labels/{split}/{name}.txt', 'w').write(line)
            n += 1
        fid += 1
    cap.release()
    return split, n

if __name__ == '__main__':
    jobs = []
    for split in ('train', 'val', 'test'):
        os.makedirs(f'{OUT}/images/{split}', exist_ok=True)
        os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)
        jobs += [(split, m, r) for m, r in json.load(open(f'{SRC}/info/{split}.json'))]
    counts = {'train': 0, 'val': 0, 'test': 0}
    with Pool(os.cpu_count()) as pool:
        for i, (split, n) in enumerate(pool.imap_unordered(convert_clip, jobs), 1):
            counts[split] += n
            if i % 25 == 0 or i == len(jobs):
                print(f'{i}/{len(jobs)} clips -- {counts}', flush=True)
    open(f'{OUT}/data.yaml', 'w').write(f'path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 1\nnames:\n  0: ball\n')

In [ ]:
# 4. Extraction des ~21 500 frames annotees (~30 min la premiere fois)
import os
if not os.path.exists('/content/yolo_rv/data.yaml'):
    !python /content/prepare.py
else:
    print('Dataset deja pret')

In [ ]:
# 5. Entrainement YOLO11s -- reprend automatiquement si une session precedente a ete coupee
import os
from ultralytics import YOLO

RUNS = f'{DRIVE}/runs'
last = f'{RUNS}/ball_rv/weights/last.pt'

if os.path.exists(last):
    print('Reprise de', last)
    YOLO(last).train(resume=True)
else:
    # yolo11s.pt = nouveau modele, poids generiques COCO (aucun lien avec ball_v1).
    # Pour des poids 100 % aleatoires, remplace par YOLO('yolo11s.yaml') (moins bon en 10 h).
    YOLO('yolo11s.pt').train(
        data='/content/yolo_rv/data.yaml',
        imgsz=1280,          # la balle fait 4-8 px a cette taille : ne pas descendre
        epochs=100,          # plafond theorique ; 'time' coupe avant
        time=10,             # heures max d'entrainement (le nombre d'epoques s'ajuste)
        batch=16,
        workers=os.cpu_count(),
        cache=False,         # 17 500 images en 1280 px : trop gros pour la RAM
        project=RUNS,
        name='ball_rv',
        patience=15,
        # Reglages pour un objet minuscule (identiques a scripts/train.py)
        scale=0.25, mosaic=1.0, close_mosaic=10,
        degrees=0.0, flipud=0.0, fliplr=0.5,
        plots=True,
    )

In [ ]:
# 6. Evaluation finale sur val et sur test (clips jamais vus a l'entrainement)
import json
best = YOLO(f'{RUNS}/ball_rv/weights/best.pt')
out = {}
for split in ('val', 'test'):
    m = best.val(data='/content/yolo_rv/data.yaml', split=split, imgsz=1280, batch=16,
                 project=RUNS, name=f'ball_rv_eval_{split}', exist_ok=True, plots=False)
    out[split] = dict(precision=float(m.box.mp), recall=float(m.box.mr),
                      mAP50=float(m.box.map50), mAP50_95=float(m.box.map),
                      ms_par_image_T4=round(sum(m.speed.values()), 1))
print(json.dumps(out, indent=2))
open(f'{RUNS}/ball_rv/final_metrics.json', 'w').write(json.dumps(out, indent=2))
print(f'\nPoids a recuperer : {RUNS}/ball_rv/weights/best.pt')
if not RUNS.startswith('/content/drive'):
    from google.colab import files
    files.download(f'{RUNS}/ball_rv/weights/best.pt')   # pas de Drive : telechargement direct